# 04 — Qwen3-VL + BERT + Graph Transformer + ViT Multimodal

Notebook này là prototype phát triển mô hình giải thích trên tập **validation** EMOTIC (PAMI). Pipeline gồm hai phần tách biệt:

1. **Nhận diện cảm xúc**: Qwen3-VL chỉ trích các dấu hiệu nhìn thấy và quan hệ; BERT mã hóa graph ngữ nghĩa; ViT mã hóa người và ảnh toàn cảnh; Graph Transformer và Multimodal Transformer hợp nhất các modality để dự đoán 26 nhãn.
2. **Giải thích có căn cứ**: loại từng modality khỏi mô hình đã huấn luyện để đo thay đổi xác suất, đóng gói evidence, rồi mới yêu cầu Qwen3-VL diễn đạt evidence thành lời.

Qwen3-VL không được xem là nguồn bằng chứng cho quyết định của emotion model. Modality ablation trong notebook là phép kiểm tra intervention sơ bộ, chưa thay thế region attribution, Integrated Gradients hay relation ablation đầy đủ.

Tất cả mẫu đều lấy từ `val` và được chia theo **ảnh** thành train/dev/eval để người trong cùng ảnh không rơi vào hai phần khác nhau. Notebook này không dùng test set.


## 1. Mount Drive, đồng bộ mã nguồn và cài dependencies

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition"
)
REPO_URL = (
    "https://github.com/minh-1105/"
    "Explainable-Contextual-Emotion-Recognition-via-"
    "Text-Guided-Chain-of-Thought-Reasoning.git"
)
CODE_ROOT = Path("/content/contextual-emotion-reasoning")

assert DRIVE_ROOT.is_dir(), f"Không tìm thấy project trên Drive: {DRIVE_ROOT}"

if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")],
    check=True,
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Drive root:", DRIVE_ROOT)
print("Code root :", CODE_ROOT)


Mounted at /content/drive
Drive root: /content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition
Code root : /content/contextual-emotion-reasoning


## 2. Cấu hình run

Qwen3-VL 4B và BERT/ViT được chạy lần lượt rồi giải phóng GPU; các feature được cache trên Drive. `RUN_ID` chỉ dùng để tiếp tục đúng cùng commit, config và prompt. Thay đổi một trong các thành phần đó phải tạo run mới (`RUN_ID = None`).


In [2]:
from datetime import datetime
import gc
import json
import os
import random
import shutil
import time
import zipfile

import numpy as np
import torch
import transformers

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("transformers:", transformers.__version__)
print("git commit:", GIT_COMMIT)

EMOTIC_ROOT = DRIVE_ROOT / "data/raw/raw/emotic"
PAMI_ROOT = EMOTIC_ROOT / "PAMI"
CVPR_ROOT = EMOTIC_ROOT / "CVPR"
ANNOTATION_PATH = PAMI_ROOT / "Annotations/Annotations/Annotations.mat"

LOCAL_EMOTIC_ROOT = Path("/content/emotic")
IMAGE_ARCHIVES = (
    (PAMI_ROOT / "pami_emotic.zip", LOCAL_EMOTIC_ROOT / "PAMI", "emotic", 23185),
    (CVPR_ROOT / "cvpr_emotic.zip", LOCAL_EMOTIC_ROOT / "CVPR", "cvpr_emotic", 18302),
)

MODEL_CONFIG = {
    "qwen_model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "bert_model_id": "google-bert/bert-base-uncased",
    "vit_backbone": "vit_b_16_imagenet1k_v1",
    "split": "val",
    "n_samples": 300,
    "seed": 42,
    "category_policy": "union",
    "max_pixels": 640 * 640,
    "max_new_tokens_cues": 320,
    "max_new_tokens_explanation": 256,
    "max_graph_nodes": 16,
    "bert_max_length": 64,
    "fusion_dim": 256,
    "graph_layers": 2,
    "fusion_layers": 2,
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "train_fraction": 0.70,
    "dev_fraction": 0.15,
    "n_explanations": 10,
    "git_commit": GIT_COMMIT,
}

RUN_ID = None  # Ví dụ: "20261104-200000" để tiếp tục đúng run cũ.


def extract_images(archive_path, extract_dir, root_name, expected_files):
    """Giải nén zip ảnh từ Drive vào ổ local của Colab, mỗi runtime một lần."""
    image_root = extract_dir / root_name
    marker = extract_dir / ".extracted"
    if marker.is_file():
        return image_root
    assert archive_path.is_file(), f"Thiếu file zip: {archive_path}"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    started = time.time()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    file_count = sum(1 for path in image_root.rglob("*") if path.is_file())
    assert file_count == expected_files, (
        f"{archive_path.name}: {file_count} file, cần {expected_files}"
    )
    marker.touch()
    print(f"Giải nén {archive_path.name}: {file_count:,} file, {time.time() - started:.0f}s")
    return image_root


assert torch.cuda.is_available(), "Hãy bật GPU: Runtime > Change runtime type > GPU"
assert ANNOTATION_PATH.is_file(), f"Thiếu annotation: {ANNOTATION_PATH}"
IMAGE_ROOTS = tuple(extract_images(*spec) for spec in IMAGE_ARCHIVES)

OUTPUT_ROOT = DRIVE_ROOT / "outputs/qwen3_multimodal"
if RUN_ID is None:
    RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
    RUN_DIR = OUTPUT_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True)
    (RUN_DIR / "config.json").write_text(
        json.dumps(MODEL_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8"
    )
else:
    RUN_DIR = OUTPUT_ROOT / RUN_ID
    saved_config = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))
    assert saved_config == MODEL_CONFIG, "Config hoặc git commit khác run cũ; hãy tạo run mới."

environment_path = RUN_DIR / "environment.txt"
environment_text = (
    f"transformers={transformers.__version__}\n"
    f"torch={torch.__version__}\n"
    f"gpu={torch.cuda.get_device_name(0)}\n"
    f"git_commit={GIT_COMMIT}\n"
)
if environment_path.is_file():
    saved_environment = environment_path.read_text(encoding="utf-8")
    assert f"git_commit={GIT_COMMIT}\n" in saved_environment
else:
    environment_path.write_text(environment_text, encoding="utf-8")
print("Run dir:", RUN_DIR)


transformers: 5.22.0
git commit: 2cb79c7c8177e1732f0bf092b18a985d2d481264
Giải nén pami_emotic.zip: 23,185 file, 92s
Giải nén cvpr_emotic.zip: 18,302 file, 36s
Run dir: /content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition/outputs/qwen3_multimodal/20261104-055316


## 3. Lấy mẫu và chia train/dev/eval theo ảnh

Mỗi người là một mẫu nhưng mọi người thuộc cùng ảnh luôn ở cùng split. `sample_id` giữ nguyên dạng `<split>:<image_index>:<person_index>`.


In [3]:
from collections import defaultdict
from src.datasets.emotic_dataset import EMOTION_LABELS, EmoticDataset

val_dataset = EmoticDataset(
    annotation_path=ANNOTATION_PATH,
    split=MODEL_CONFIG["split"],
    image_roots=IMAGE_ROOTS,
    category_policy=MODEL_CONFIG["category_policy"],
    strict_images=True,
)
sample_indices = sorted(
    random.Random(MODEL_CONFIG["seed"]).sample(
        range(len(val_dataset)), MODEL_CONFIG["n_samples"]
    )
)
sample_ids = [val_dataset.samples[index].sample_id for index in sample_indices]

indices_by_image = defaultdict(list)
for index in sample_indices:
    indices_by_image[val_dataset.samples[index].relative_image_path].append(index)
image_ids = sorted(indices_by_image)
random.Random(MODEL_CONFIG["seed"]).shuffle(image_ids)

n_images = len(image_ids)
train_end = round(n_images * MODEL_CONFIG["train_fraction"])
dev_end = train_end + round(n_images * MODEL_CONFIG["dev_fraction"])
image_split = {
    "train": set(image_ids[:train_end]),
    "dev": set(image_ids[train_end:dev_end]),
    "eval": set(image_ids[dev_end:]),
}
split_sample_ids = {
    split: [
        val_dataset.samples[index].sample_id
        for image_id in image_ids if image_id in members
        for index in indices_by_image[image_id]
    ]
    for split, members in image_split.items()
}
assert not (image_split["train"] & image_split["dev"])
assert not (image_split["train"] & image_split["eval"])
assert not (image_split["dev"] & image_split["eval"])
assert set().union(*map(set, split_sample_ids.values())) == set(sample_ids)

samples_payload = {
    "sample_ids": sample_ids,
    "splits": split_sample_ids,
    "split_unit": "relative_image_path",
}
samples_path = RUN_DIR / "samples.json"
if samples_path.is_file():
    assert json.loads(samples_path.read_text(encoding="utf-8")) == samples_payload
else:
    samples_path.write_text(
        json.dumps(samples_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

for split in ("train", "dev", "eval"):
    print(f"{split}: {len(split_sample_ids[split])} người / {len(image_split[split])} ảnh")


train: 210 người / 195 ảnh
dev: 43 người / 42 ảnh
eval: 47 người / 42 ảnh


## 4. Prompt trích cue và prompt giải thích

Prompt đầu không đưa danh sách cảm xúc hay ground truth cho Qwen3-VL, nhằm tránh rò nhãn vào feature văn bản. Prompt sau chỉ nhận prediction và evidence do emotion model tạo ra.


In [4]:
import re
from PIL import ImageDraw

CUE_PROMPT = """The target person is the one inside the red box.
Extract only visible information that may help a separate emotion-recognition model.
Do not name or infer any emotion. Do not use facts that are not visible.
Return one JSON object with exactly these keys in this order:
- \"person_cues\": a non-empty list of short visible face, pose, gesture, clothing or action cues.
- \"context_cues\": a non-empty list of short visible scene, object or other-person cues.
- \"relations\": a list of short visible relations involving the target, such as \"target facing person_2\". Use [] if none is visible.
- \"situation\": one short sentence describing only what is visibly happening.
Every list element must be a plain string."""

EXPLANATION_PROMPT = """You are given the output and intervention evidence of an emotion-recognition model.
Explain why the model produced its prediction using only the supplied evidence.
Do not add a cue, causal claim or emotion that is absent from the evidence.
State which modality mattered most and cite probability changes when available.
Return JSON only as {\"explanation\": \"...\"}.

Evidence package:
{evidence}
"""

prompts_payload = {"cue_extraction": CUE_PROMPT, "grounded_explanation": EXPLANATION_PROMPT}
prompts_path = RUN_DIR / "prompts.json"
if prompts_path.is_file():
    assert json.loads(prompts_path.read_text(encoding="utf-8")) == prompts_payload
else:
    prompts_path.write_text(
        json.dumps(prompts_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def mark_target(image, bbox):
    marked = image.copy()
    width = max(3, round(max(marked.size) / 150))
    ImageDraw.Draw(marked).rectangle(
        [float(value) for value in bbox], outline=(255, 0, 0), width=width
    )
    return marked


def parse_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        payload, _ = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None


def cue_schema_ok(payload):
    keys = ["person_cues", "context_cues", "relations", "situation"]
    if not isinstance(payload, dict) or list(payload) != keys:
        return False
    for key in keys[:3]:
        if not isinstance(payload[key], list) or not all(
            isinstance(value, str) and value.strip() for value in payload[key]
        ):
            return False
    return (
        bool(payload["person_cues"])
        and bool(payload["context_cues"])
        and isinstance(payload["situation"], str)
        and bool(payload["situation"].strip())
    )


## 5. Qwen3-VL: trích graph ngữ nghĩa nhìn thấy

In [5]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration


def load_qwen():
    capability = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if capability[0] >= 8 else torch.float16
    processor = AutoProcessor.from_pretrained(
        MODEL_CONFIG["qwen_model_id"], max_pixels=MODEL_CONFIG["max_pixels"]
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_CONFIG["qwen_model_id"],
        torch_dtype=dtype,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    ).to("cuda").eval()
    return processor, model


@torch.inference_mode()
def ask_qwen(processor, model, image, prompt, max_new_tokens):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    inputs = inputs.to("cuda")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()
    return text, generated.shape[1] >= max_new_tokens


qwen_processor, qwen_model = load_qwen()
print("Đã nạp", MODEL_CONFIG["qwen_model_id"])


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Đã nạp Qwen/Qwen3-VL-4B-Instruct


## 6. Smoke test rồi sinh cue cho toàn bộ mẫu

Chạy smoke test và đọc output thô. Nếu schema sai, dừng và sửa prompt; prompt thay đổi đồng nghĩa với run mới.


In [6]:
for index in sample_indices[:2]:
    item = val_dataset[index]
    image = mark_target(item["context_image"], item["bbox"].tolist())
    raw, truncated = ask_qwen(
        qwen_processor, qwen_model, image, CUE_PROMPT,
        MODEL_CONFIG["max_new_tokens_cues"],
    )
    payload = parse_json_object(raw)
    print("=" * 80)
    print("sample_id:", item["sample_id"], "| schema:", cue_schema_ok(payload),
          "| truncated:", truncated)
    print(raw)


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


sample_id: val:1:0 | schema: True | truncated: False
{
  "person_cues": [
    "standing",
    "holding curtain",
    "wearing light-colored shirt",
    "wearing light-colored pants"
  ],
  "context_cues": [
    "hotel room",
    "bed with white linens",
    "person lying on bed",
    "curtains",
    "suitcase",
    "lamp"
  ],
  "relations": [
    "target facing person_2"
  ],
  "situation": "A person is standing by a window holding a curtain while another person lies on a bed."
}
sample_id: val:10:0 | schema: True | truncated: False
{
  "person_cues": [
    "wearing bright green reflective jacket",
    "standing in water",
    "holding object in hand",
    "facing away from camera"
  ],
  "context_cues": [
    "flooded street",
    "blue car with brake lights on",
    "woman reaching toward car",
    "another car in background"
  ],
  "relations": [
    "target facing blue car",
    "target near woman"
  ],
  "situation": "A person in a reflective jacket stands in floodwater near a ca

In [ ]:
from tqdm.auto import tqdm

cues_path = RUN_DIR / "qwen_cues.jsonl"
done = set()
if cues_path.is_file():
    done = {
        json.loads(line)["sample_id"]
        for line in cues_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }
todo = [
    index for index in sample_indices
    if val_dataset.samples[index].sample_id not in done
]
print(f"Qwen cues: {len(done)} đã có, còn {len(todo)}")

with cues_path.open("a", encoding="utf-8") as file:
    for index in tqdm(todo, desc="Qwen3-VL cues"):
        item = val_dataset[index]
        image = mark_target(item["context_image"], item["bbox"].tolist())
        started = time.time()
        raw, truncated = ask_qwen(
            qwen_processor, qwen_model, image, CUE_PROMPT,
            MODEL_CONFIG["max_new_tokens_cues"],
        )
        payload = parse_json_object(raw)
        record = {
            "sample_id": item["sample_id"],
            "relative_image_path": item["relative_image_path"],
            "bbox": item["bbox"].tolist(),
            "parse_ok": payload is not None,
            "schema_ok": cue_schema_ok(payload),
            "truncated": truncated,
            "response": payload,
            "raw": raw,
            "seconds": round(time.time() - started, 2),
        }
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()

cue_records_by_id = {
    record["sample_id"]: record
    for record in map(json.loads, cues_path.read_text(encoding="utf-8").splitlines())
}
assert set(cue_records_by_id) == set(sample_ids), "Cue còn thiếu; chạy lại cell này."
print("schema rate:", np.mean([cue_records_by_id[sid]["schema_ok"] for sid in sample_ids]))

del qwen_model, qwen_processor
gc.collect()
torch.cuda.empty_cache()


Qwen cues: 0 đã có, còn 300


Qwen3-VL cues:   0%|          | 0/300 [00:00<?, ?it/s]

## 7. BERT: mã hóa các node của graph ngữ nghĩa

Mỗi cue, relation và situation là một node có type riêng. BERT được đóng băng và chỉ dùng tạo embedding; Graph Transformer phía sau học tương tác giữa các node.


In [ ]:
from transformers import BertModel, BertTokenizerFast

NODE_TYPE = {"person": 0, "context": 1, "relation": 2, "situation": 3}


def semantic_nodes(record):
    payload = record["response"] if record["schema_ok"] else None
    if payload is None:
        return [("situation", "No reliable visible cues were extracted.")]
    nodes = []
    nodes.extend(("person", text) for text in payload["person_cues"])
    nodes.extend(("context", text) for text in payload["context_cues"])
    nodes.extend(("relation", text) for text in payload["relations"])
    nodes.append(("situation", payload["situation"]))
    return nodes[:MODEL_CONFIG["max_graph_nodes"]]


bert_cache = RUN_DIR / "bert_graph_features.npz"
if not bert_cache.is_file():
    tokenizer = BertTokenizerFast.from_pretrained(MODEL_CONFIG["bert_model_id"])
    bert_model = BertModel.from_pretrained(MODEL_CONFIG["bert_model_id"]).to("cuda").eval()
    for parameter in bert_model.parameters():
        parameter.requires_grad_(False)

    node_lists = [semantic_nodes(cue_records_by_id[sid]) for sid in sample_ids]
    flat = [
        (row, column, kind, text)
        for row, nodes in enumerate(node_lists)
        for column, (kind, text) in enumerate(nodes)
    ]
    hidden_size = bert_model.config.hidden_size
    node_features = np.zeros(
        (len(sample_ids), MODEL_CONFIG["max_graph_nodes"], hidden_size), dtype=np.float32
    )
    node_mask = np.zeros(
        (len(sample_ids), MODEL_CONFIG["max_graph_nodes"]), dtype=np.bool_
    )
    node_types = np.zeros_like(node_mask, dtype=np.int64)

    batch_size = 64
    for start in tqdm(range(0, len(flat), batch_size), desc="BERT nodes"):
        batch = flat[start:start + batch_size]
        tokens = tokenizer(
            [entry[3] for entry in batch],
            padding=True,
            truncation=True,
            max_length=MODEL_CONFIG["bert_max_length"],
            return_tensors="pt",
        ).to("cuda")
        with torch.inference_mode():
            embedding = bert_model(**tokens).last_hidden_state[:, 0].float().cpu().numpy()
        for entry, vector in zip(batch, embedding):
            row, column, kind, _ = entry
            node_features[row, column] = vector
            node_mask[row, column] = True
            node_types[row, column] = NODE_TYPE[kind]

    np.savez_compressed(
        bert_cache,
        sample_ids=np.asarray(sample_ids),
        node_features=node_features,
        node_mask=node_mask,
        node_types=node_types,
    )
    del bert_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

bert_data = np.load(bert_cache, allow_pickle=False)
assert bert_data["sample_ids"].tolist() == sample_ids
print("BERT graph features:", bert_data["node_features"].shape)


## 8. ViT: mã hóa người và ảnh toàn cảnh

ViT ImageNet được đóng băng. Feature được cache để việc huấn luyện Graph/Multimodal Transformer nhẹ và có thể lặp lại trên T4.


In [ ]:
import torch.nn as nn
from torchvision.models import ViT_B_16_Weights, vit_b_16

vit_cache = RUN_DIR / "vit_features.npz"
if not vit_cache.is_file():
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    transform = weights.transforms()
    vit_model = vit_b_16(weights=weights)
    vit_model.heads = nn.Identity()
    vit_model = vit_model.to("cuda").eval()
    for parameter in vit_model.parameters():
        parameter.requires_grad_(False)

    person_features, context_features, labels = [], [], []
    feature_batch_size = 16
    for start in tqdm(range(0, len(sample_indices), feature_batch_size), desc="ViT features"):
        indices = sample_indices[start:start + feature_batch_size]
        items = [val_dataset[index] for index in indices]
        person = torch.stack([transform(item["person_image"]) for item in items]).to("cuda")
        context = torch.stack([transform(item["context_image"]) for item in items]).to("cuda")
        with torch.inference_mode():
            person_features.append(vit_model(person).float().cpu().numpy())
            context_features.append(vit_model(context).float().cpu().numpy())
        labels.extend(item["labels"].numpy() for item in items)

    np.savez_compressed(
        vit_cache,
        sample_ids=np.asarray(sample_ids),
        person_features=np.concatenate(person_features),
        context_features=np.concatenate(context_features),
        labels=np.asarray(labels, dtype=np.float32),
    )
    del vit_model
    gc.collect()
    torch.cuda.empty_cache()

vit_data = np.load(vit_cache, allow_pickle=False)
assert vit_data["sample_ids"].tolist() == sample_ids
print("Person/context ViT:", vit_data["person_features"].shape)


## 9. Graph Transformer và Multimodal Transformer

Graph Transformer dùng self-attention trên graph cue đầy đủ; node type thay cho vị trí tuần tự. Multimodal Transformer hợp nhất token người, bối cảnh và graph. Đây là kiến trúc prototype huấn luyện trên feature đã đóng băng.


In [ ]:
class SemanticGraphTransformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_layers):
        super().__init__()
        self.projection = nn.Linear(input_dim, model_dim)
        self.type_embedding = nn.Embedding(len(NODE_TYPE), model_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=8, dim_feedforward=model_dim * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(model_dim)

    def forward(self, features, mask, node_types):
        tokens = self.projection(features) + self.type_embedding(node_types)
        tokens = self.encoder(tokens, src_key_padding_mask=~mask)
        weights = mask.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * weights).sum(1) / weights.sum(1).clamp_min(1)
        return self.norm(pooled)


class EmotionMultimodalTransformer(nn.Module):
    def __init__(self, visual_dim, text_dim, model_dim, graph_layers, fusion_layers, n_classes):
        super().__init__()
        self.person_projection = nn.Linear(visual_dim, model_dim)
        self.context_projection = nn.Linear(visual_dim, model_dim)
        self.graph = SemanticGraphTransformer(text_dim, model_dim, graph_layers)
        self.cls = nn.Parameter(torch.zeros(1, 1, model_dim))
        self.modality_embedding = nn.Parameter(torch.randn(1, 4, model_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=8, dim_feedforward=model_dim * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.fusion = nn.TransformerEncoder(layer, num_layers=fusion_layers)
        self.head = nn.Sequential(nn.LayerNorm(model_dim), nn.Dropout(0.2), nn.Linear(model_dim, n_classes))

    def forward(self, person, context, nodes, node_mask, node_types, drop_modality=None):
        graph = self.graph(nodes, node_mask, node_types)
        batch_size = person.shape[0]
        tokens = torch.stack(
            [
                self.cls.expand(batch_size, -1, -1)[:, 0],
                self.person_projection(person),
                self.context_projection(context),
                graph,
            ],
            dim=1,
        ) + self.modality_embedding
        drop_index = {"person": 1, "context": 2, "graph": 3}.get(drop_modality)
        if drop_index is not None:
            tokens[:, drop_index] = 0
        return self.head(self.fusion(tokens)[:, 0])


## 10. Huấn luyện trên train, chọn checkpoint/threshold trên dev, báo cáo eval

`eval` ở đây là phần giữ lại của PAMI validation, không phải EMOTIC test. Checkpoint chọn bằng dev mAP và threshold từng nhãn cũng chỉ chọn trên dev.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from src.evaluation.emotion_metrics import multilabel_metrics, select_per_class_thresholds

id_to_row = {sample_id: row for row, sample_id in enumerate(sample_ids)}
rows = {
    split: np.asarray([id_to_row[sid] for sid in ids], dtype=np.int64)
    for split, ids in split_sample_ids.items()
}

person_all = torch.from_numpy(vit_data["person_features"]).float()
context_all = torch.from_numpy(vit_data["context_features"]).float()
nodes_all = torch.from_numpy(bert_data["node_features"]).float()
node_mask_all = torch.from_numpy(bert_data["node_mask"]).bool()
node_types_all = torch.from_numpy(bert_data["node_types"]).long()
labels_all = torch.from_numpy(vit_data["labels"]).float()


def loader_for(split, shuffle=False):
    index = torch.from_numpy(rows[split]).long()
    dataset = TensorDataset(
        person_all[index], context_all[index], nodes_all[index],
        node_mask_all[index], node_types_all[index], labels_all[index], index,
    )
    return DataLoader(
        dataset, batch_size=MODEL_CONFIG["batch_size"], shuffle=shuffle,
        num_workers=0, pin_memory=True,
    )


train_loader = loader_for("train", shuffle=True)
dev_loader = loader_for("dev")
eval_loader = loader_for("eval")

fusion_model = EmotionMultimodalTransformer(
    visual_dim=person_all.shape[-1],
    text_dim=nodes_all.shape[-1],
    model_dim=MODEL_CONFIG["fusion_dim"],
    graph_layers=MODEL_CONFIG["graph_layers"],
    fusion_layers=MODEL_CONFIG["fusion_layers"],
    n_classes=len(EMOTION_LABELS),
).to("cuda")

train_targets = labels_all[torch.from_numpy(rows["train"])].numpy()
positives = train_targets.sum(0)
negatives = len(train_targets) - positives
pos_weight = np.divide(negatives, positives, out=np.ones_like(negatives), where=positives > 0)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.from_numpy(np.clip(pos_weight, 1.0, 10.0)).float().to("cuda")
)
optimizer = torch.optim.AdamW(
    fusion_model.parameters(), lr=MODEL_CONFIG["learning_rate"],
    weight_decay=MODEL_CONFIG["weight_decay"],
)


@torch.inference_mode()
def predict(loader, drop_modality=None):
    fusion_model.eval()
    probabilities, targets, output_rows = [], [], []
    for person, context, nodes, mask, types, labels, batch_rows in loader:
        logits = fusion_model(
            person.to("cuda"), context.to("cuda"), nodes.to("cuda"),
            mask.to("cuda"), types.to("cuda"), drop_modality=drop_modality,
        )
        probabilities.append(logits.sigmoid().cpu().numpy())
        targets.append(labels.numpy())
        output_rows.append(batch_rows.numpy())
    return np.concatenate(probabilities), np.concatenate(targets), np.concatenate(output_rows)


best_map = -float("inf")
best_state = None
history = []
for epoch in range(1, MODEL_CONFIG["epochs"] + 1):
    fusion_model.train()
    losses = []
    for person, context, nodes, mask, types, labels, _ in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits = fusion_model(
            person.to("cuda"), context.to("cuda"), nodes.to("cuda"),
            mask.to("cuda"), types.to("cuda"),
        )
        loss = criterion(logits, labels.to("cuda"))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    dev_probabilities, dev_targets, _ = predict(dev_loader)
    dev_metrics = multilabel_metrics(
        dev_targets, dev_probabilities, class_names=EMOTION_LABELS
    )
    row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "dev_map": dev_metrics["map"]}
    history.append(row)
    print(row)
    if np.isfinite(dev_metrics["map"]) and dev_metrics["map"] > best_map:
        best_map = dev_metrics["map"]
        best_state = {key: value.detach().cpu() for key, value in fusion_model.state_dict().items()}

assert best_state is not None
fusion_model.load_state_dict(best_state)
checkpoint = {
    "model_state_dict": best_state,
    "model_config": MODEL_CONFIG,
    "class_names": list(EMOTION_LABELS),
    "best_dev_map": best_map,
}
torch.save(checkpoint, RUN_DIR / "model.pt")

dev_probabilities, dev_targets, dev_rows = predict(dev_loader)
thresholds = select_per_class_thresholds(dev_targets, dev_probabilities)
eval_probabilities, eval_targets, eval_rows = predict(eval_loader)
dev_metrics = multilabel_metrics(
    dev_targets, dev_probabilities, thresholds=thresholds, class_names=EMOTION_LABELS
)
eval_metrics = multilabel_metrics(
    eval_targets, eval_probabilities, thresholds=thresholds, class_names=EMOTION_LABELS
)
metrics_payload = {
    "scope": "PAMI val split by image; no test data",
    "best_dev_map": best_map,
    "thresholds": thresholds.tolist(),
    "dev": dev_metrics,
    "eval": eval_metrics,
    "qwen_cues": {
        "parse_rate": float(np.mean([cue_records_by_id[sid]["parse_ok"] for sid in sample_ids])),
        "schema_rate": float(np.mean([cue_records_by_id[sid]["schema_ok"] for sid in sample_ids])),
        "truncated_rate": float(np.mean([cue_records_by_id[sid]["truncated"] for sid in sample_ids])),
    },
    "history": history,
}
(RUN_DIR / "metrics.json").write_text(
    json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
np.savez_compressed(
    RUN_DIR / "predictions_eval.npz",
    sample_ids=np.asarray([sample_ids[row] for row in eval_rows]),
    targets=eval_targets,
    probabilities=eval_probabilities,
    thresholds=thresholds,
)
print(json.dumps({"dev": dev_metrics, "eval": eval_metrics}, ensure_ascii=False, indent=2))


## 11. Modality ablation → evidence package

Mỗi modality lần lượt bị thay bằng token 0. Độ giảm xác suất là evidence intervention ở mức modality; không diễn giải nó như quan hệ nhân quả ở mức pixel hay semantic relation.


In [ ]:
ablated_probabilities = {}
for modality in ("person", "context", "graph"):
    ablated_probabilities[modality], ablated_targets, ablated_rows = predict(
        eval_loader, drop_modality=modality
    )
    assert np.array_equal(ablated_targets, eval_targets)
    assert np.array_equal(ablated_rows, eval_rows)

eval_predictions = eval_probabilities >= thresholds
evidence_packages = []
for position, row in enumerate(eval_rows):
    predicted_indices = np.flatnonzero(eval_predictions[position])
    if len(predicted_indices) == 0:
        predicted_indices = np.asarray([int(eval_probabilities[position].argmax())])
    emotion_evidence = []
    for label_index in predicted_indices:
        emotion_evidence.append({
            "emotion": EMOTION_LABELS[label_index],
            "probability": round(float(eval_probabilities[position, label_index]), 4),
            "probability_without_person": round(float(ablated_probabilities["person"][position, label_index]), 4),
            "probability_without_context": round(float(ablated_probabilities["context"][position, label_index]), 4),
            "probability_without_graph": round(float(ablated_probabilities["graph"][position, label_index]), 4),
        })
    sample_id = sample_ids[row]
    cue_record = cue_records_by_id[sample_id]
    evidence_packages.append({
        "sample_id": sample_id,
        "prediction": emotion_evidence,
        "visible_cues": cue_record["response"] if cue_record["schema_ok"] else None,
        "intervention": "Each modality token was replaced by zero; values report the resulting probability.",
    })

(RUN_DIR / "evidence_packages.json").write_text(
    json.dumps(evidence_packages, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã tạo", len(evidence_packages), "evidence package.")


## 12. Qwen3-VL diễn đạt explanation có căn cứ

Chỉ sinh một tập ví dụ cố định để kiểm tra định tính. Mỗi explanation lưu cùng evidence đầu vào để có thể truy vết từng phát biểu.


In [ ]:
qwen_processor, qwen_model = load_qwen()
explanations_path = RUN_DIR / "explanations.jsonl"
selected_evidence = sorted(evidence_packages, key=lambda row: row["sample_id"])[
    :MODEL_CONFIG["n_explanations"]
]
done = set()
if explanations_path.is_file():
    done = {
        json.loads(line)["sample_id"]
        for line in explanations_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }
index_by_id = {val_dataset.samples[index].sample_id: index for index in sample_indices}

with explanations_path.open("a", encoding="utf-8") as file:
    for evidence in tqdm(selected_evidence, desc="Grounded explanations"):
        if evidence["sample_id"] in done:
            continue
        item = val_dataset[index_by_id[evidence["sample_id"]]]
        image = mark_target(item["context_image"], item["bbox"].tolist())
        prompt = EXPLANATION_PROMPT.format(
            evidence=json.dumps(evidence, ensure_ascii=False, indent=2)
        )
        raw, truncated = ask_qwen(
            qwen_processor, qwen_model, image, prompt,
            MODEL_CONFIG["max_new_tokens_explanation"],
        )
        payload = parse_json_object(raw)
        record = {
            "sample_id": evidence["sample_id"],
            "evidence": evidence,
            "response": payload,
            "parse_ok": isinstance(payload, dict) and isinstance(payload.get("explanation"), str),
            "truncated": truncated,
            "raw": raw,
        }
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()

del qwen_model, qwen_processor
gc.collect()
torch.cuda.empty_cache()
print("Hoàn tất run:", RUN_DIR)


## 13. Giới hạn và bước tiếp theo

- Qwen cue extraction vẫn có thể sai; cần đánh giá thủ công cue/relations trên một mẫu cố định.
- Graph hiện là fully connected semantic graph. Bước tiếp theo là scene graph có node/object/relation được định danh và relation ablation thực sự.
- Modality ablation bằng token 0 có distribution shift; cần bổ sung region occlusion, Integrated Gradients và visual-token attribution.
- Kết quả eval chỉ là phần giữ lại của validation để phát triển. Sau khi kiến trúc và protocol ổn định mới khóa cấu hình và đánh giá đúng một lần trên test trong Notebook 03 hoặc notebook đánh giá riêng.
- Cần so sánh ablation `ViT only`, `ViT + BERT`, `ViT + graph`, và full multimodal bằng cùng image split trước khi kết luận thành phần nào có ích.
